# Exploratory Data Analysis (EDA)
Generate data and perform basic exploration.

In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
N = 3000
GROSS_MARGIN = 0.35

def generate_tier(n, tier_name, config):
    data = {
        'tenure_months':     np.random.randint(config['tenure'][0], config['tenure'][1], n),
        'total_orders':      np.random.randint(config['orders'][0], config['orders'][1], n),
        'total_spend':       np.clip(
            np.random.lognormal(mean=np.log(config['spend_mean']),
                                sigma=config['spend_sigma'], size=n),
            config['spend_clip'][0], config['spend_clip'][1]
        ),
        'recency_days':      np.random.randint(config['recency'][0], config['recency'][1], n),
        'age':               np.random.randint(config['age'][0], config['age'][1], n),
        'income_bracket':    np.random.choice(config['income'], size=n),
        'nps_score':         np.random.randint(config['nps'][0], config['nps'][1], n),
        'online_ratio':      np.random.uniform(config['online'][0], config['online'][1], n),
        'return_rate':       np.random.uniform(config['return_rate'][0], config['return_rate'][1], n),
        'support_tickets':   np.random.randint(config['tickets'][0], config['tickets'][1], n),
        'discount_usage':    np.random.uniform(config['discount'][0], config['discount'][1], n),
        'category_preference': np.random.choice(
            config['categories'], size=n,
            p=config.get('cat_probs', None)
        ),
        'region':            np.random.choice(['North', 'South', 'East', 'West'], size=n),
        'gender':            np.random.choice(['M', 'F'], size=n),
        'clv_tier':          tier_name,
    }
    return pd.DataFrame(data)

tier1_config = {
    'tenure': (24, 84), 'orders': (40, 200), 'spend_mean': 8000, 'spend_sigma': 0.4,
    'spend_clip': (2500, 15000), 'recency': (1, 30), 'age': (30, 65), 'income': [3, 4, 4, 4],
    'nps': (8, 11), 'online': (0.3, 0.8), 'return_rate': (0.01, 0.08), 'tickets': (0, 5),
    'discount': (0.0, 0.15), 'categories': ['Electronics', 'Luxury', 'Home'], 'cat_probs': [0.4, 0.4, 0.2]
}
tier2_config = {
    'tenure': (6, 36), 'orders': (12, 60), 'spend_mean': 1500, 'spend_sigma': 0.45,
    'spend_clip': (800, 2500), 'recency': (15, 90), 'age': (25, 55), 'income': [2, 3, 3],
    'nps': (6, 10), 'online': (0.4, 0.9), 'return_rate': (0.05, 0.15), 'tickets': (1, 8),
    'discount': (0.1, 0.35), 'categories': ['Clothing', 'Sports', 'Tech'], 'cat_probs': [0.4, 0.3, 0.3]
}
tier3_config = {
    'tenure': (12, 48), 'orders': (4, 20), 'spend_mean': 450, 'spend_sigma': 0.5,
    'spend_clip': (200, 800), 'recency': (60, 200), 'age': (20, 70), 'income': [1, 2, 2],
    'nps': (4, 8), 'online': (0.2, 0.6), 'return_rate': (0.1, 0.3), 'tickets': (3, 15),
    'discount': (0.3, 0.7), 'categories': ['Grocery', 'Pharmacy', 'Clothing'], 'cat_probs': None
}
tier4_config = {
    'tenure': (1, 24), 'orders': (1, 8), 'spend_mean': 80, 'spend_sigma': 0.6,
    'spend_clip': (10, 200), 'recency': (180, 365), 'age': (18, 75), 'income': [1, 1, 2],
    'nps': (1, 7), 'online': (0.1, 0.5), 'return_rate': (0.2, 0.5), 'tickets': (2, 20),
    'discount': (0.4, 1.0), 'categories': ['Grocery', 'Pharmacy'], 'cat_probs': None
}

df_champions   = generate_tier(600,  'Champions',   tier1_config)
df_growers     = generate_tier(900,  'Growers',     tier2_config)
df_atrisk      = generate_tier(800,  'At-Risk',     tier3_config)
df_hibernating = generate_tier(700,  'Hibernating', tier4_config)

df = pd.concat([df_champions, df_growers, df_atrisk, df_hibernating], ignore_index=True)
df['customer_id'] = [f"CUST_{i+1:05d}" for i in range(len(df))]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df['avg_order_value'] = (df['total_spend'] / df['total_orders']).clip(lower=1.0)
df['tenure_years'] = df['tenure_months'] / 12.0
df['purchase_frequency'] = (df['total_orders'] / df['tenure_years']).clip(upper=365.0)
df['recency_score'] = pd.qcut(df['recency_days'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)
df['frequency_score'] = pd.qcut(df['total_orders'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)
df['monetary_score'] = pd.qcut(df['total_spend'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop').astype(int)
df['rfm_score'] = df['recency_score'] + df['frequency_score'] + df['monetary_score']

base_retention = (df['recency_score'] + df['frequency_score']) / 10.0
noise = np.random.normal(0, 0.05, size=len(df))
df['retention_rate'] = (base_retention + noise).clip(0.05, 0.95)
df['churn_rate'] = 1.0 - df['retention_rate']

raw_clv = (df['avg_order_value'] * df['purchase_frequency'] * GROSS_MARGIN) / df['churn_rate']
clv_noise = np.random.lognormal(0, 0.15, size=len(df))
df['clv_12month'] = (raw_clv * clv_noise).clip(5.0, 50000.0).round(2)

df['clv_segment'] = pd.cut(df['clv_12month'], bins=[0, 200, 800, 2500, 50001], labels=['Low', 'Medium', 'High', 'Very High'])

os.makedirs("../data", exist_ok=True)
df.to_csv("../data/customers.csv", index=False)
df.head()


## Basic Exploration

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.histplot(df['clv_12month'], bins=50, kde=True)
plt.title('Distribution of 12-Month CLV')
plt.xlabel('CLV ($)')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='total_spend', y='clv_12month', hue='clv_segment')
plt.title('Total Spend vs 12-Month CLV')
plt.show()